<a href="https://colab.research.google.com/github/ian-menachery/ECON3916-Statistical-Machine-Learning/blob/main/%5BLab_12_%5D_OLS%2C_Hedonic_Pricing%2C_and_RMSE_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

# Step 1: Ingestion from external source
url = 'Zillow_ZHVI_2026_Micro.csv'
df = pd.read_csv(url)
print(df)

     Home_Value  Square_Footage  Property_Age  Distance_to_Transit  \
0     329705.74          1941.0           5.5                 6.45   
1     183343.63          1364.3          35.2                 2.15   
2     354551.73          2386.9          52.4                 0.75   
3     325773.17          2192.1          50.2                 5.25   
4     359743.12          3069.8          66.5                12.69   
..          ...             ...           ...                  ...   
995   389368.45          2744.4           4.1                10.49   
996   380314.99          2512.3           8.7                 6.18   
997   230005.36          1765.8          73.6                 6.57   
998   360943.75          2021.1          16.6                 5.48   
999   194390.17          1263.3          78.5                13.47   

    School_District_Rating  
0                Excellent  
1                  Average  
2                     Good  
3                Excellent  
4             

In [5]:
# Step 2: Defining the formula
# Utilizing the R-style patsy formula interface allows for elegant, readable model specification
formula = 'Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit + School_District_Rating'

In [9]:
# Step 3: Fitting the model and printing the summary
model = smf.ols(formula=formula, data=df)
results = model.fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     542.5
Date:                Mon, 16 Mar 2026   Prob (F-statistic):          2.81e-309
Time:                        20:02:18   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.416e+04
Df Residuals:                     993   BIC:                         2.419e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [10]:
# Step 4: Generating predictions
# We use the results object to predict Home_Value based on the original dataframe features
y_pred = results.predict(df)

In [11]:
model_rmse = rmse(df['Home_Value'], y_pred)
print(f"\nThe Predictive RMSE is: ${model_rmse:,.2f}")


The Predictive RMSE is: $42,316.69


In [12]:
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd

# --- PREPARATION ---
# 1. Extract Fitted Values and Residuals directly from the statsmodels 'results' object
# 'fittedvalues' are the model's predictions (y_hat)
# 'resid' is the difference between the actual value and the prediction (y - y_hat)
df['predictions'] = results.fittedvalues
df['residuals'] = results.resid

# 2. Logic for Outlier Detection (2 Standard Deviations)
# We calculate the standard deviation of the residuals to find the "error threshold"
resid_std = df['residuals'].std()
df['is_outlier'] = df['residuals'].abs() > (2 * resid_std)

# 3. Define a discrete color map for visual clarity
# Highlights structural failures in Crimson while keeping the rest Neutral
color_map = {True: 'crimson', False: '#34495e'}

# --- DASHBOARD CONSTRUCTION ---
fig = px.scatter(
    df,
    x='predictions',
    y='residuals',
    color='is_outlier',
    color_discrete_map=color_map,
    hover_data=['Home_Value', 'Square_Footage'], # Adds context for forensic deep-dives
    title="<b>2026 Zillow OLS Residual Forensics</b><br><sup>Detecting Structural Breaks and Price Elasticity</sup>",
    labels={'predictions': 'Model Predicted Value ($)', 'residuals': 'Residual Error ($)'},
    template='plotly_white'
)

# 4. Adding the Horizontal Zero-Line (The "Perfection" Line)
# If the model were 100% accurate, every dot would live on this line
fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.7)

# 5. UI/UX Tweaks
fig.update_layout(
    showlegend=False,
    xaxis_title="Predicted Home Value",
    yaxis_title="Residual Error (Over/Under Prediction)"
)

fig.show()